## **Import & Setup**

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

## **Load the Data**

In [5]:
# Input paths
BUSINESS_DATA_PATH = 'C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\04. Joining Datasets\\3. Joining - Business - Neighborhood Data\\Extracted CSV Files\\business_data.csv'
NEIGHBORHOOD_LIST_PATH = 'C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\03. Base Datasets\\3. Data -  Smaller Spatial Units\\1. Neighborhoods\\Extracted CSV Files\\neighborhoods_enriched.csv'

# Column names
NACE_COL = 'NACE Code'
NACE_DESC_COL = 'NACE Description (EN)'
NEIGHBORHOOD_COL = 'Neighborhood'

# Output paths
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)
MATRIX_PATH = OUTPUT_DIR / 'interaction_matrix_raw.parquet'
METADATA_PATH = OUTPUT_DIR / 'matrix_metadata.json'

# Construction parameters
MIN_BUSINESSES_FOR_CF = 2   # NACE classes below this are flagged (not dropped) for eval-time filtering

In [6]:
df = pd.read_csv(BUSINESS_DATA_PATH)
df = df.dropna(subset=[NACE_COL, NEIGHBORHOOD_COL]).copy()

print(f'Loaded {len(df):,} usable business records')
print(f'Unique NACE classes: {df[NACE_COL].nunique()}')
print(f'Unique neighbourhoods in business data: {df[NEIGHBORHOOD_COL].nunique()}')

Loaded 3,882 usable business records
Unique NACE classes: 306
Unique neighbourhoods in business data: 39


In [ ]:
# The business data only contains 39 neighbourhoods; we need the full 48
# so the matrix includes zero-business neighbourhoods as empty columns.
nbhd_df = pd.read_csv(NEIGHBORHOOD_LIST_PATH)

all_neighborhoods = sorted(nbhd_df['Neighborhood'].unique().tolist())
print(f'Canonical neighbourhood universe: {len(all_neighborhoods)} (expected 48)')

# Sanity: which neighbourhoods appear in the canonical list but have zero businesses?
present_in_data = set(df[NEIGHBORHOOD_COL].unique())
zero_business_nbhds = [n for n in all_neighborhoods if n not in present_in_data]
print(f'Zero-business neighbourhoods: {len(zero_business_nbhds)}')
print(zero_business_nbhds)

Canonical neighbourhood universe: 48 (expected 48)
Zero-business neighbourhoods: 9
['Agios Apostolos o Neos', 'Agios Lavrentios', 'Agios Minas', 'Aidini', 'Glafyra', 'Katohori', 'Platanidia', 'Stagiates', 'Xrisi Akti Panagias']


## **Build the raw interaction matrix**

In [8]:
matrix_long = (
    df.groupby([NACE_COL, NEIGHBORHOOD_COL])
      .size()
      .reset_index(name='count')
)

matrix = (
    matrix_long
    .pivot(index=NACE_COL, columns=NEIGHBORHOOD_COL, values='count')
    .fillna(0)
    .astype(int)
)

# Ensure ALL 48 neighbourhoods are columns (adds empty columns for zero-business ones)
matrix = matrix.reindex(columns=all_neighborhoods, fill_value=0)

print(f'Matrix shape: {matrix.shape}  (expected 306 × 48)')

Matrix shape: (306, 48)  (expected 306 × 48)


In [9]:
# 1. Total businesses in the matrix must equal usable record count
total_in_matrix = int(matrix.values.sum())
print(f'Total businesses in matrix: {total_in_matrix:,}')
print(f'Usable records loaded:      {len(df):,}')
assert total_in_matrix == len(df), 'Mismatch! Some records lost or double-counted.'

# 2. Shape check
assert matrix.shape[1] == 48, f'Expected 48 neighbourhoods, got {matrix.shape[1]}'

# 3. Sparsity
nonzero = int((matrix > 0).sum().sum())
density = nonzero / matrix.size
print(f'Non-zero cells: {nonzero:,} / {matrix.size:,}  (density {density:.2%})')

# 4. Zero-business neighbourhoods (columns that are all zero)
empty_cols = matrix.columns[(matrix == 0).all()].tolist()
print(f'All-zero neighbourhood columns: {len(empty_cols)}')

print('\nAll validation checks passed.')

Total businesses in matrix: 3,882
Usable records loaded:      3,882
Non-zero cells: 1,702 / 14,688  (density 11.59%)
All-zero neighbourhood columns: 9

All validation checks passed.


## **Add Metadata**

In [11]:
nace_counts = df.groupby(NACE_COL).size()
low_signal_nace = nace_counts[nace_counts < MIN_BUSINESSES_FOR_CF].index.tolist()

# NACE code -> English description mapping
nace_desc_map = (
    df.drop_duplicates(subset=[NACE_COL])
      .set_index(NACE_COL)[NACE_DESC_COL]
      .astype(str)
      .to_dict()
)

metadata = {
    'built_at': datetime.now().isoformat(),
    'source_file': str(BUSINESS_DATA_PATH),
    'value_type': 'raw_counts',
    'shape': {'n_nace': int(matrix.shape[0]), 'n_neighborhoods': int(matrix.shape[1])},
    'total_businesses': total_in_matrix,
    'density': round(density, 4),
    'min_businesses_for_cf': MIN_BUSINESSES_FOR_CF,
    'low_signal_nace_classes': [str(c) for c in low_signal_nace],   # flag, don't drop
    'n_low_signal_nace': len(low_signal_nace),
    'zero_business_neighborhoods': zero_business_nbhds,
    'nace_descriptions': {str(k): v for k, v in nace_desc_map.items()},
}

print(f'NACE classes with <{MIN_BUSINESSES_FOR_CF} businesses (flagged): {len(low_signal_nace)}')
print(f'Zero-business neighbourhoods: {len(zero_business_nbhds)}')

NACE classes with <2 businesses (flagged): 56
Zero-business neighbourhoods: 9


## **Save the Artifacts**

In [ ]:
matrix.to_parquet(MATRIX_PATH)
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'Saved matrix to:   {MATRIX_PATH}')
print(f'Saved metadata to: {METADATA_PATH}')

Saved matrix to:   ..\data\interaction_matrix_raw.parquet
Saved metadata to: ..\data\matrix_metadata.json


: 